In [1]:
# Import the usual libraries and variables.
%run standard_imports.py

# Install a pip package in the current Jupyter kernel.
import sys
!{sys.executable} -m pip install --user pypika

DEPRECATION: Python 2.7 will reach the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 won't be maintained after that date. A future version of pip will drop support for Python 2.7.


In [2]:
import datetime
from pypika import MySQLQuery, Table, Order, functions as fn

start_date = datetime.date(2019, 1, 1)
end_date = datetime.date(2020, 1, 1)

charge = Table("charge")
merchant_app_charge = Table("merchant_app_charge")
merchant_app = Table("merchant_app")
developer_app = Table("developer_app")
developer = Table("developer")

gross_revenue = (fn.Sum(charge.amount)/100).as_("gross_revenue")
gross_developer_portion = (fn.Sum(fn.Coalesce(charge.developer_portion,
                                             charge.amount * 0.7))/100).as_("gross_developer_portion")
gross_clover_portion = (fn.Sum(charge.amount - fn.Coalesce(charge.developer_portion,
                                                          charge.amount * 0.7))/100).as_("gross_clover_portion")

q = MySQLQuery.from_(charge).join(
    merchant_app_charge
).on(
    merchant_app_charge.charge_id == charge.id 
).join(
    merchant_app
).on(
    merchant_app.id == merchant_app_charge.merchant_app_id
).join(
    developer_app
).on(
    developer_app.id == merchant_app.app_id
).join(
    developer
).on(
    developer.id == developer_app.developer_id
).select(
    developer.uuid,
    developer.name,
    gross_revenue,
    gross_developer_portion,
    gross_clover_portion
).where(
    (developer.name != "Clover") &
    (developer.is_rev_share_flat_rate == 0)
).where(
    (charge.system_type == "INFOLEASE") &
    (charge.status.isin(["DISBURSED", "BILLED"])) &
    (charge.currency == "USD")
).where(
    charge.created_time[start_date:end_date]
).groupby(developer.id) \
.orderby(gross_clover_portion, order=Order.desc)

query = q.get_sql()
print(query)

db = Db("~/.clover/p801.cfg")
df = pd.read_sql(query, con=db.conn)
db.close()

df

SELECT `developer`.`uuid`,`developer`.`name`,SUM(`charge`.`amount`)/100 `gross_revenue`,SUM(COALESCE(`charge`.`developer_portion`,`charge`.`amount`*0.7))/100 `gross_developer_portion`,SUM(`charge`.`amount`-COALESCE(`charge`.`developer_portion`,`charge`.`amount`*0.7))/100 `gross_clover_portion` FROM `charge` JOIN `merchant_app_charge` ON `merchant_app_charge`.`charge_id`=`charge`.`id` JOIN `merchant_app` ON `merchant_app`.`id`=`merchant_app_charge`.`merchant_app_id` JOIN `developer_app` ON `developer_app`.`id`=`merchant_app`.`app_id` JOIN `developer` ON `developer`.`id`=`developer_app`.`developer_id` WHERE `developer`.`name`<>'Clover' AND `developer`.`is_rev_share_flat_rate`=0 AND `charge`.`system_type`='INFOLEASE' AND `charge`.`status` IN ('DISBURSED','BILLED') AND `charge`.`currency`='USD' AND `charge`.`created_time` BETWEEN '2019-01-01' AND '2020-01-01' GROUP BY `developer`.`id` ORDER BY `gross_clover_portion` DESC


,uuid,name,gross_revenue,gross_developer_portion,gross_clover_portion
0,2SW5NA5C30G10,Homebase,772430.64,540797.147,231633.493
1,R9XNECD2A8VKP,Infuse,462994.22,324121.314,138872.906
2,3GVN1P855JXEA,Abreeze Technology,457940.13,320648.962,137291.168
3,DK22NG4RNCM7W,Seven Spaces,412933.44,289260.850,123672.590
4,QCRD5T0ZX1HSG,DAVO Technologies,306448.02,214545.219,91902.801
5,DJJ1DMZS8AP42,Menufy,238006.25,166608.020,71398.230
6,1Z9K2PP9WGRKT,"Shopventory, Inc.",187454.37,131218.614,56235.756
7,H0MPC0J3SVFY6,4 Leaf Labs,183747.84,128645.473,55102.367
8,DF48AHKSK0A94,TapMango Inc.,166243.63,116370.730,49872.900
9,FXP6KXWMEVCB4,Zaytech,163615.10,114552.667,49062.433


In [3]:
import datetime
import calendar

def annualize_ytd(value):
    current = datetime.datetime.now()
    day_of_year = current.timetuple().tm_yday
    total_days = 366 if calendar.isleap(current.year) else 365
    year_progress = float(day_of_year)/float(total_days)
    return value/year_progress

df["annualized_gross_revenue"] = df["gross_revenue"].map(lambda x: round(annualize_ytd(x)))
df

,uuid,name,gross_revenue,gross_developer_portion,gross_clover_portion,annualized_gross_revenue
0,2SW5NA5C30G10,Homebase,772430.64,540797.147,231633.493,2104009.0
1,R9XNECD2A8VKP,Infuse,462994.22,324121.314,138872.906,1261141.0
2,3GVN1P855JXEA,Abreeze Technology,457940.13,320648.962,137291.168,1247374.0
3,DK22NG4RNCM7W,Seven Spaces,412933.44,289260.850,123672.590,1124781.0
4,QCRD5T0ZX1HSG,DAVO Technologies,306448.02,214545.219,91902.801,834728.0
5,DJJ1DMZS8AP42,Menufy,238006.25,166608.020,71398.230,648301.0
6,1Z9K2PP9WGRKT,"Shopventory, Inc.",187454.37,131218.614,56235.756,510603.0
7,H0MPC0J3SVFY6,4 Leaf Labs,183747.84,128645.473,55102.367,500507.0
8,DF48AHKSK0A94,TapMango Inc.,166243.63,116370.730,49872.900,452828.0
9,FXP6KXWMEVCB4,Zaytech,163615.10,114552.667,49062.433,445668.0


In [4]:
DIAMOND_REV_THRESHOLD = 2400000
GOLD_REV_THRESHOLD    =  600000
SILVER_REV_THRESHOLD  =  250000
BRONZE_REV_THRESHOLD  =   50000

def categorize_by_rev(value):
    if value > DIAMOND_REV_THRESHOLD:
        return "Diamond"
    if value > GOLD_REV_THRESHOLD:
        return "Gold"
    if value > SILVER_REV_THRESHOLD:
        return "Silver"
    if value > BRONZE_REV_THRESHOLD:
        return "Bronze"
    else:
        return "Public"


df["support_tier"] = df["annualized_gross_revenue"].map(lambda x: categorize_by_rev(x))
df

,uuid,name,gross_revenue,gross_developer_portion,gross_clover_portion,annualized_gross_revenue,support_tier
0,2SW5NA5C30G10,Homebase,772430.64,540797.147,231633.493,2104009.0,Gold
1,R9XNECD2A8VKP,Infuse,462994.22,324121.314,138872.906,1261141.0,Gold
2,3GVN1P855JXEA,Abreeze Technology,457940.13,320648.962,137291.168,1247374.0,Gold
3,DK22NG4RNCM7W,Seven Spaces,412933.44,289260.850,123672.590,1124781.0,Gold
4,QCRD5T0ZX1HSG,DAVO Technologies,306448.02,214545.219,91902.801,834728.0,Gold
5,DJJ1DMZS8AP42,Menufy,238006.25,166608.020,71398.230,648301.0,Gold
6,1Z9K2PP9WGRKT,"Shopventory, Inc.",187454.37,131218.614,56235.756,510603.0,Silver
7,H0MPC0J3SVFY6,4 Leaf Labs,183747.84,128645.473,55102.367,500507.0,Silver
8,DF48AHKSK0A94,TapMango Inc.,166243.63,116370.730,49872.900,452828.0,Silver
9,FXP6KXWMEVCB4,Zaytech,163615.10,114552.667,49062.433,445668.0,Silver
